# Second judge — cross-family Φ over the Batch API

Scores the seven main conditions' validation outputs with an OpenAI rater, so the
register metric Φ no longer rests on a single judge.

---
## 1 — Setup

API-only: no GPU, no torch.

In [2]:
!git pull

Already up to date.


In [3]:
!pip install -q openai==2.41.1 PyYAML==6.0.3 python-dotenv==1.2.2 numpy scipy


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import getpass, logging, os
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)

In [5]:
%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


---
## 2 — Pre-flight

Each assertion guards a way this run could corrupt the primary Φ or produce an
invalid comparison. Do not relax one to make the cell pass.

In [6]:
import json, pathlib, yaml
from src.eval.judge import judge_results_path, judge_segment_dir, template_digest
from src.eval.judge_batch import read_state, state_path
from src.eval.judge_ci import MAIN_CONDITIONS

CONFIG  = 'configs/judge_eval_gpt.yaml'
SPLIT   = 'val'
CONDS   = MAIN_CONDITIONS            # the seven main conditions
RESULTS = pathlib.Path('results')

cfg = yaml.safe_load(pathlib.Path(CONFIG).read_text())
judge, TAG = cfg['judge'], cfg['tag']

# -- a genuinely different family, or there is no cross-check
assert judge['provider'] == 'openai', judge['provider']
assert TAG, 'tag must be set, or this run overwrites the primary judge artefacts'
assert judge['model'] and judge['model'] != 'terra', \
    'set the confirmed API model id in the config -- a wrong id fails the batch'

# -- reasoning off is the whole cost argument; assert it rather than trust it
assert judge.get('reasoning_effort') == 'none', judge.get('reasoning_effort')

# -- identical frozen rubric
assert cfg['template_file'] == 'prompts/judge_eval.txt', cfg['template_file']
DIGEST = template_digest(pathlib.Path(cfg['template_file']).read_text())

# -- artefacts must not collide with the primary judge's
out_b, dir_b = judge_results_path(RESULTS, SPLIT, TAG), judge_segment_dir(RESULTS, SPLIT, TAG)
out_a = judge_results_path(RESULTS, SPLIT, None)
assert out_b != out_a
assert out_a.exists(), 'the primary judge results are missing; nothing to compare against'
a = json.loads(out_a.read_text())
models_a = sorted({v['model'] for v in a.values() if isinstance(v, dict)})
assert judge['model'] not in models_a, 'judge B is the same model as judge A'

# -- all seven conditions present, equal length, identical segments in identical order
rows = {c: [json.loads(x) for x in pathlib.Path(f'outputs/{c}_{SPLIT}.jsonl')
            .read_text().splitlines() if x.strip()] for c in CONDS}
n_eval = len(rows[CONDS[0]])
for c in CONDS:
    assert len(rows[c]) == n_eval, f'{c}: {len(rows[c])} rows, expected {n_eval}'
    assert [r['input'] for r in rows[c]] == [r['input'] for r in rows[CONDS[0]]], \
        f'{c} segments diverge from {CONDS[0]}'

print(f'conditions   : {len(CONDS)}  {CONDS}')
print(f'segments each: {n_eval}   (test split sealed)')
print(f'judge A      : {models_a}')
print(f'judge B      : {judge["model"]}  tag={TAG}  reasoning={judge["reasoning_effort"]}')
print(f'rubric       : {cfg["template_file"]} [{DIGEST}]  (same file as judge A)')
print(f'writes to    : {out_b}  and  {dir_b}/')

inflight = read_state(state_path(RESULTS, SPLIT, TAG))
print(f'in-flight batches: {inflight or "none"}')
if inflight:
    print('  -> a previous submission is still running. Re-running section 5 RESUMES')
    print('     polling it. It does NOT resubmit, so you are not billed twice.')

/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


conditions   : 7  ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full', 'peft', 'commercial_haiku']
segments each: 1323   (test split sealed)
judge A      : ['claude-haiku-4-5']
judge B      : gpt-5.6-terra  tag=gpt  reasoning=none
rubric       : prompts/judge_eval.txt [ffd6dad41acb0512]  (same file as judge A)
writes to    : results/judge_gpt_val.json  and  results/judge_gpt_val_segments/
in-flight batches: none


---
## 3 — Pilot

In [8]:
!python3 manage.py judge_batch --conditions zeroshot --split {SPLIT} \
    --config {CONFIG} --limit 25 --poll_interval 15

judge gpt-5.6-terra [batch]  tag=gpt  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 25 segments for zeroshot with gpt-5.6-terra ...
  submitting 25 requests ...
  submitted batch batch_6a72cf024fdc8190a237dd2ad3e5a2af
  [validating] 0/0 completed
  [validating] 0/0 completed
  [validating] 0/0 completed
  [validating] 0/0 completed
  [validating] 0/0 completed
  [in_progress] 0/25 completed
  [in_progress] 22/25 completed
  [in_progress] 22/25 completed
  [in_progress] 22/25 completed
  [completed] 25/25 completed
  batch completed: wrote 25 segment(s)
  zeroshot         Φ 3.920  (coverage 100%)

pilot only (--limit 25): segment cache written to results/judge_gpt_val_segments, results/judge_gpt_val.json deliberately NOT written. Re-run without --limit for the full split.
Judge usage: {'calls': 25, 'prompt_tokens': 11536, 'completion_tokens': 1030, 'cost_usd': 0.0177}
Wrote results/judge_gpt_val_usage.json  (cumulative $0.02)


In [10]:
u = json.loads((RESULTS / f'judge_{TAG}_{SPLIT}_usage.json').read_text())
s = u['session']
if s['calls'] == 0:
    print('no calls billed this session (cache already complete)')
else:
    per = {k: s[k]/s['calls'] for k in ('prompt_tokens','completion_tokens','cost_usd')}
    print(f"pilot: {s['calls']} calls, {s['prompt_tokens']:,} in / {s['completion_tokens']:,} out")
    print(f"       {per['prompt_tokens']:.0f} in / {per['completion_tokens']:.0f} out per call")
    if per['completion_tokens'] > 150:
        print('  !! far above the ~60 expected -- the model is likely still emitting')
        print('     reasoning tokens. Re-check reasoning_effort before the full run.')
    if not u['priced']:
        print('  !! no pricing configured; cost_usd is a floor of 0, not real spend')
    else:
        print(f"\nPROJECTED FULL PASS: {calls:,} calls  ~${per['cost_usd']*calls:.2f}")


pilot: 25 calls, 11,536 in / 1,030 out
       461 in / 41 out per call

PROJECTED FULL PASS: 9,261 calls  ~$6.56


---
## 4 — Full pass

In [11]:
CONDS_ARG = ' '.join(CONDS)
!python3 manage.py judge_batch --conditions {CONDS_ARG} --split {SPLIT} --config {CONFIG}

judge gpt-5.6-terra [batch]  tag=gpt  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1323 segments for zeroshot with gpt-5.6-terra ...
  resuming: 25/1323 already scored
  submitting 1298 requests ...
  submitted batch batch_6a72d05269fc8190988c3b0d9e31983f
  [validating] 0/0 completed
  [in_progress] 0/1298 completed
  [in_progress] 409/1298 completed
  [in_progress] 744/1298 completed
  [in_progress] 1107/1298 completed
  [in_progress] 1202/1298 completed
  [finalizing] 1298/1298 completed
  [finalizing] 1298/1298 completed
  [completed] 1298/1298 completed
  batch completed: wrote 1298 segment(s), 23 unscored
  zeroshot         Φ 3.648  (coverage 98%)
Judging 1323 segments for random_fewshot with gpt-5.6-terra ...
  submitting 1323 requests ...
  submitted batch batch_6a72d14b92ac8190bb4906491262bf40
  [validating] 0/0 completed
  [validating] 0/0 completed
  [validating] 0/0 completed
  [in_progress] 0/1323 completed
  [in_progress] 117/1323 completed
  [in_progress] 99

In [13]:
b = json.loads(out_b.read_text())
print(f"{'condition':<18}{'n':>6}{'coverage':>10}{'Phi_B':>8}{'Phi_A':>8}{'B-A':>8}")
for c in CONDS:
    rb, ra = b.get(c), a.get(c)
    if not rb:
        print(f'{c:<18}  MISSING'); continue
    d = rb['mean'] - ra['mean'] if ra and ra.get('mean') and rb.get('mean') else float('nan')
    print(f"{c:<18}{rb['n']:>6}{rb['coverage']:>10.4f}{rb['mean']:>8.3f}"
          f"{(ra['mean'] if ra else float('nan')):>8.3f}{d:>+8.3f}")

errs = {c: sum('error' in json.loads(x)
               for x in (dir_b / f'{c}.jsonl').read_text().splitlines() if x.strip())
        for c in CONDS if (dir_b / f'{c}.jsonl').exists()}


condition              n  coverage   Phi_B   Phi_A     B-A
zeroshot            1323    0.9826   3.648   2.546  +1.102
random_fewshot      1323    0.9803   3.633   2.633  +1.000
knn_fewshot         1323    0.9887   3.679   2.748  +0.931
afsp_margin         1323    0.9849   3.667   2.763  +0.904
afsp_full           1323    0.9902   3.707   2.791  +0.916
peft                1323    0.9872   3.613   2.744  +0.869
commercial_haiku    1323    0.9887   3.981   3.333  +0.648


---
## 5 — Standalone Φ table

In [14]:
!python3 manage.py judge_ci --tag {TAG} --split {SPLIT} --n_resamples 10000


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: gpt-5.6-terra  [tag gpt]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition         class      n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
-------------------------------------------------------------------------------------------------------------
1     commercial_haiku  reference  1308  3.9809  [3.9386, 4.0222]  0.760  1.000         1 (1.000)   1.00     
2     afsp_full         study      1310  3.7069  [3.6608, 3.7525]  0.841  0.874         2 (0.874)   2.15     
3     knn_fewshot       study      1308  3.6789  [3.6345, 3.7231]  0.815  0.584         3 (0.584)   3.30     
4     afsp_margin       study      1303  3.6669  [3.6236, 3.7109]  0.808  0.510         4 (0.510)   4.03     
5     zeroshot          study      1300  3.6477  [3.6029, 3.6917]  0.810  0.459         5 (0.459)   5.06     
6     random_fewsh

In [15]:
# The same table for the primary judge, for context. Separate artefact, not merged.
!python3 manage.py judge_ci --split {SPLIT} --n_resamples 10000


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: claude-haiku-4-5  [tag (none)]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition         class      n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
-------------------------------------------------------------------------------------------------------------
1     commercial_haiku  reference  1323  3.3333  [3.2880, 3.3787]  0.843  1.000         1 (1.000)   1.00     
2     afsp_full         study      1323  2.7906  [2.7385, 2.8420]  0.948  0.838         2 (0.838)   2.21     
3     afsp_margin       study      1323  2.7627  [2.7135, 2.8133]  0.915  0.530         3 (0.530)   3.36     
4     knn_fewshot       study      1322  2.7481  [2.6982, 2.7995]  0.928  0.429         4 (0.429)   4.17     
5     peft              study      1323  2.7438  [2.6893, 2.7997]  1.033  0.526         5 (0.526)   4.26     
6     random

---
## 6 — Judge–judge agreement


In [16]:
!python3 manage.py judge_agreement --tag_b {TAG} --split {SPLIT} --n_resamples 10000


Judge-judge agreement  (split=val, resamples=10000, seed=42, 95% percentile CIs)
  judge A: claude-haiku-4-5  [tag (none)]
  judge B: gpt-5.6-terra  [tag gpt]
  same frozen rubric verified by digest: False
  ! at least one judge's results predate template digest recording; that both raters read prompts/judge_eval.txt is asserted, not verified

Coverage (segments parsed by each rater)
condition         n_total  n_a   n_b   n_both
---------------------------------------------
zeroshot          1323     1323  1300  1300  
random_fewshot    1323     1323  1297  1297  
knn_fewshot       1323     1322  1308  1307  
afsp_margin       1323     1323  1303  1303  
afsp_full         1323     1323  1310  1310  
peft              1323     1323  1306  1306  
commercial_haiku  1323     1323  1308  1308  

Rater agreement -- study_only
condition       n     Phi_A  Phi_B  A-B     ci95              qwk     qwk_ci            rho     exact  adj  
----------------------------------------------------------

In [17]:
rep = json.loads((RESULTS / f'judge_agreement_{TAG}_{SPLIT}.json').read_text())
pooled = rep['rater_agreement']['study_only']['pooled']
print(f"pooled n={pooled['n']}  qwk={pooled['qwk']['kappa']:+.3f}"
      f"  rho={pooled['spearman']['rho']:+.3f}"
      f"  exact={pooled['exact_agreement']:.1%}  adjacent={pooled['adjacent_agreement']:.1%}")
print(f"severity offset A-B = {pooled['offset']['diff']:+.3f}"
      f" [{pooled['offset']['ci_low']:+.3f}, {pooled['offset']['ci_high']:+.3f}]")
print(f"identical system ranking: {rep['condition_ordering']['study_only']['identical_ranking']}")

print('\nContrasts by stability:')
for v_name in ('RATER-DEPENDENT', 'both separate', 'neither separates'):
    names = [k for k, v in rep['contrast_replication']['contrasts'].items()
             if (('both separate' if v['both_separate'] else
                  'neither separates' if v['neither_separates'] else
                  'RATER-DEPENDENT') == v_name)]
    print(f'  {v_name:<18} {names or "-"}')
flipped = [k for k, v in rep['contrast_replication']['contrasts'].items() if not v['same_sign']]
print(f'\nsign flips between raters: {flipped or "none"}')

pooled n=7823  qwk=+0.384  rho=+0.592  exact=27.8%  adjacent=77.2%
severity offset A-B = -0.950 [-0.968, -0.932]
identical system ranking: False

Contrasts by stability:
  RATER-DEPENDENT    ['random_fewshot - zeroshot', 'knn_fewshot - zeroshot', 'afsp_margin - zeroshot', 'peft - zeroshot', 'peft - afsp_full']
  both separate      ['afsp_full - zeroshot']
  neither separates  ['afsp_full - knn_fewshot', 'afsp_margin - knn_fewshot', 'afsp_full - afsp_margin']

sign flips between raters: ['random_fewshot - zeroshot', 'peft - zeroshot', 'afsp_margin - knn_fewshot']
